In [2]:
import asyncio
import aiohttp
import json
import ssl
import nest_asyncio

# Enable asyncio in Jupyter
nest_asyncio.apply()

async def make_async_request(prompt):
    # Create SSL context that doesn't verify certificate
    ssl_context = ssl.create_default_context()
    ssl_context.check_hostname = False
    ssl_context.verify_mode = ssl.CERT_NONE

    url = "https://10.1.13.168:8443/mbdevai/openai/deployments/azure-chat-engine-gpt-4o-base/chat/completions?api-version=2024-02-15-preview&api-key=REPLACE_WITH_LOCAL_SECRET"
    
    payload = {
        "stream": False,
        "messages": [
            {
                "role": "user",
                "content": prompt
            }
        ],
        "temperature": 0
    }

    headers = {
        'Content-Type': 'application/json'
    }

    async with aiohttp.ClientSession() as session:
        async with session.post(
            url,
            headers=headers,
            json=payload,
            ssl=ssl_context
        ) as response:
            result = await response.json()
            return result['choices'][0]['message']

In [3]:
# For Jupyter, you can run this directly
async def process_prompts(prompts):
    tasks = [make_async_request(prompt) for prompt in prompts]
    responses = await asyncio.gather(*tasks)
    return responses

In [4]:
# Example usage
prompts = [
    "xin chào",
    "đánh nhau không, dkm",
    "hi em yêu"
]

# Run the async code
responses = asyncio.run(process_prompts(prompts))

# Print results
for prompt, response in zip(prompts, responses):
    print(f"Prompt: {prompt}")
    print(f"Response: {response['content']}\n")

Prompt: xin chào
Response: Xin chào! Tôi có thể giúp gì cho bạn hôm nay?

Prompt: đánh nhau không, dkm
Response: Xin lỗi, nhưng tôi không thể giúp bạn với yêu cầu này. Nếu bạn có bất kỳ câu hỏi nào khác hoặc cần hỗ trợ về một vấn đề khác, hãy cho tôi biết!

Prompt: hi em yêu
Response: Chào bạn! Có điều gì mình có thể giúp bạn không?



In [5]:
from tqdm.asyncio import tqdm_asyncio
from tqdm import tqdm


def create_evaluation_prompt(row):
    prompt = f"""Given the following:
Question: {row['question']}
Student Answer: {row['answer']}
Correct Answer: {row['truth_answer']}

Focus ONLY on the main intent/purpose of the question. 
If the student's answer addresses the core concept asked in the question and aligns with the key point in the correct answer, mark it as correct. 
Ignore additional details or minor variations.
Respond with ONLY ONE WORD: either 'correct' or 'incorrect'."""
    return prompt


async def process_batch(rows, batch_size=10):
    prompts = [create_evaluation_prompt(row) for _, row in rows.iterrows()]
    total_batches = (len(prompts) + batch_size - 1) // batch_size
    
    results = []
    pbar = tqdm(total=len(prompts), desc="Processing evaluations")
    
    for i in range(0, len(prompts), batch_size):
        batch = prompts[i:i + batch_size]
        batch_results = await asyncio.gather(*[make_async_request(prompt) for prompt in batch])
        results.extend(batch_results)
        pbar.update(len(batch))
    
    pbar.close()
    return results

In [6]:
import pandas as pd

df = pd.read_csv(r'C:\Users\thanglq2\Downloads\big_result__800_8k.csv', encoding='utf-8')

In [7]:
# Process all rows and get evaluations
evaluations = await process_batch(df)

# Add evaluations to the dataframe
df['ai_evaluation'] = evaluations

# Save the updated dataframe
print("Saving results...")
df.to_csv('results_evaluated.csv', index=False)
print("Evaluations completed and saved to results_evaluated.csv")

Processing evaluations:   0%|          | 0/448 [00:00<?, ?it/s]

Processing evaluations: 100%|██████████| 448/448 [05:57<00:00,  1.25it/s]

Saving results...
Evaluations completed and saved to results_evaluated.csv


In [8]:
df[df['ai_evaluation'] == {'content': 'incorrect', 'role': 'assistant'}]

,question,truth_answer,answer,ref,ai_evaluation
32,thẻ jcb credit có ngày chốt sao kê là ngày nào,"Chào bạn! 😊\n\nTheo thông tin từ tài liệu, **n...","Traceback (most recent call last):\n File ""C:...",NaN,"{'content': 'incorrect', 'role': 'assistant'}"
47,phí rút tiền qua thẻ tín dụng là bao nhiêu,Chào bạn! 😊\n\nVề **phí rút tiền qua thẻ tín d...,Chào bạn! 😊\n\nVề **phí rút tiền qua thẻ tín d...,"['57ff188d-1370-4ebb-8b1a-bc66bc5a99b2', '5d63...","{'content': 'incorrect', 'role': 'assistant'}"
48,mã pin của thẻ tín dụng có bao nhiêu số,Chào bạn! 🌟\n\nMã PIN của thẻ tín dụng tại Ngâ...,Chào bạn! 🌟\n\nMã PIN của thẻ tín dụng tại MB ...,"['6d6cbc88-507f-42cb-8385-45cc2d41df68', '19dd...","{'content': 'incorrect', 'role': 'assistant'}"
57,cách tính phí phạt chậm trả thẻ tín dụng jcb ...,Chào bạn! 🌟\n\nVề **cách tính phí phạt chậm tr...,Chào bạn! 🌟\n\nVề **cách tính phí phạt chậm tr...,"['c5fd3912-9b37-4a38-85a7-9377aa901d74', '26bb...","{'content': 'incorrect', 'role': 'assistant'}"
61,có thể thay đổi hình thức trích nợ tự động của...,"Chào bạn! 😊\n\nVề câu hỏi của bạn, bạn hoàn to...",Error: 502,NaN,"{'content': 'incorrect', 'role': 'assistant'}"
99,phí xử lý nội tệ của thẻ jcb ultimate là bao ...,"Chào bạn! 😊\n\nHiện tại, mình không tìm thấy t...",Chào bạn! 🌟\n\nPhí xử lý giao dịch nội tệ của ...,"['3b9d7773-5ffa-4c34-a98c-54247884715d', 'beba...","{'content': 'incorrect', 'role': 'assistant'}"
107,phí xử lý ngoại tệ của thẻ jcb ultimate là bao...,Chào bạn! 😊\n\nPhí xử lý ngoại tệ của thẻ **JC...,Chào bạn! 😊\n\nPhí xử lý giao dịch nước ngoài ...,"['3b9d7773-5ffa-4c34-a98c-54247884715d', 'beba...","{'content': 'incorrect', 'role': 'assistant'}"
128,phí chuyển đổi ngoại tệ thẻ hi green,Chào bạn! 😊\n\nVề **phí chuyển đổi ngoại tệ** ...,Chào bạn! 😊\n\nVề **phí chuyển đổi ngoại tệ** ...,"['2f373593-1185-4789-b794-ae9d2ab57bf5', '4beb...","{'content': 'incorrect', 'role': 'assistant'}"
132,Khi KH đã đặt mua thẻ JCB Hi Sky thì có được h...,Chào bạn! 🌟\n\nVề việc **hủy đơn đặt mua thẻ J...,"Chào bạn! 🌞\n\nHiện tại, **khách hàng không th...","['e7d68076-6ba7-4b1a-8e0c-ab0bb911d75c', '68d2...","{'content': 'incorrect', 'role': 'assistant'}"
137,Thẻ Master Hi LOL đang có bao nhiêu mẫu the,Chào bạn! 😊\n\nThẻ **Master Hi League Of Legen...,Chào bạn! 😊\n\nThẻ **Master Hi League Of Legen...,"['bfacce48-be6c-4155-b605-e42d4e7283d6', 'de0a...","{'content': 'incorrect', 'role': 'assistant'}"


In [17]:
tdf.to_csv("big_result_incorrect.csv")

In [21]:
import requests
import json

url = "http://semantic-search-api.tanzu-dev.mbbank.com.vn/get_chunk_with_chunkids"

payload = json.dumps({
  "chunk_ids": """['b6d46211-428d-4b4b-b40f-e2da043448a0', '27a0b7f2-d862-4ea9-8c01-47f0b1a3d8ab', '8b968a58-e0be-4076-aac5-42e7e1aae4a4', 'e984983e-1338-41a0-beeb-d3eea19d0d53', '9a91e1b3-8162-491e-8cee-b3dabcbea7ad', '572e6b08-306d-483d-ae34-8c2810c41116', 'c4a01046-9b6f-4c49-97f9-6ae7e11dfb15', '019d8a1b-e8ec-4418-940d-de408418f567', '1f5dfe46-68e1-41e1-97c4-eb43ad7ed58e', '08f85848-012f-472a-a3bd-bb3a810b5f8a', '6c06b1b6-475b-42f9-aee4-63c8a0b6d628', '8663802f-76b7-44e7-88a0-df5a55f0ff04']"""
})
headers = {
  'Content-Type': 'application/json'
}

response = requests.request("POST", url, headers=headers, data=payload)

for msg in response.json():
    print(msg)
    print('--------------------------------')

Tên tài liệu chứa chunk: 18. Dịch vụ thanh toán giao dịch quảng cáo bằng thẻ
 Nội dung chunk: . Dịch vụ thanh toán giao dịch quảng cáo bằng thẻ. 1.. Đối tượng áp dụng. Áp dụng đối với tất cả các KHCN sử dụng sản phẩm thẻ Visa Debit và thẻ đa năng (Visa/Master/JCB) với mục đích thanh toán giao dịch quảng cáo;. Giao dịch quảng cáo là các giao dịch tại merchant category code: 7311 (mã MCC được cập nhật từng thời kỳ), ví dụ như các merchant: facebook, google, tiktok…. 2.. Mức phí: 220.000 VNĐ / thẻ (đã bao gồm VAT). 3.. Cơ chế thu phí. Phí dịch vụ được thu 1 lần duy nhất theo từng thẻ (căn cứ theo số thẻ).. Phí dịch vụ được thu tự động từ tài khoản thanh toán của khách hàng khi khách hàng có nhu cầu “mở/on” dịch vụ thanh toán giao dịch quảng cáo.. Trường hợp thu phí không thành công thì dịch vụ thanh toán giao dịch quảng cáo ở trạng thái “đóng/off”.. Chi tiết cơ chế thu phí theo Phụ lục 01 đính kèm.. 4.. Cơ chế ưu đãi. Khách hàng sẽ được hoàn toàn bộ mức phí 220.000 VNĐ khi đáp ứng các điề